# SwiGLU — Paper-to-Code Mock (Colab)

**Paper:** GLU Variants Improve Transformer (Noam Shazeer, 2020) — https://arxiv.org/abs/2002.05202

Mock (~40 min). Fill in the `SwiGLUFFN` stub, run the param-matched SwiGLU-vs-ReLU demo on a **neutral** target, then the sanity checks. Reference solution in the last cell.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(0)

## 1. Implement SwiGLU (YOUR TASK)

SwiGLU FFN: `down( Swish(x W_gate) * (x W_up) )`, with Swish = SiLU = `z*sigmoid(z)`.
Use THREE bias-free linears (gate, up, down). To match a standard `4d` FFN's params,
scale the hidden size by 2/3, i.e. `hidden = 8d/3`.

In [ ]:
def swiglu_hidden(d_model, ffn_mult=4):
    # Param-matched hidden for a 3-matrix gated FFN: (2/3)*ffn_mult*d, rounded to mult of 8.
    h = int(2 / 3 * ffn_mult * d_model)
    return (h + 7) // 8 * 8


class SwiGLUFFN(nn.Module):
    def __init__(self, d_model, hidden=None, ffn_mult=4):
        super().__init__()
        if hidden is None:
            hidden = swiglu_hidden(d_model, ffn_mult)
        # TODO: three bias-free linears: gate (d->h), up (d->h), down (h->d)
        raise NotImplementedError("fill me in")

    def forward(self, x):
        # TODO: return down( silu(gate(x)) * up(x) )
        raise NotImplementedError("fill me in")


class ReLUFFN(nn.Module):
    """Standard Transformer FFN baseline: down(ReLU(up(x)))."""

    def __init__(self, d_model, hidden=None, ffn_mult=4):
        super().__init__()
        if hidden is None:
            hidden = ffn_mult * d_model
        self.up = nn.Linear(d_model, hidden, bias=False)
        self.down = nn.Linear(hidden, d_model, bias=False)

    def forward(self, x):
        return self.down(F.relu(self.up(x)))

## 2. Demonstrate the mechanism — honestly (param-matched, NEUTRAL target)
Fit a **neutral** random-MLP teacher (favors neither block) with SwiGLU (hidden=8d/3) vs ReLU (hidden=4d) at matched params.
Honest result: SwiGLU does **not** win on this toy — it can even lose, since the 2/3 split narrows the hidden layer. The real benefit is an at-scale perplexity improvement a toy can't show (like AdamW). **Don't** rig it with a gated-structured target — that's circular.

In [ ]:
d_model = 32
n_train, n_test = 2048, 4096

# NEUTRAL target: a fixed random MLP teacher (tanh) — no affinity for either FFN.
# (A SwiGLU-shaped target like (silu(x@G)*(x@A))@B would RIG the comparison.)
torch.manual_seed(0)
teacher = nn.Sequential(
    nn.Linear(d_model, 64), nn.Tanh(),
    nn.Linear(64, 64), nn.Tanh(),
    nn.Linear(64, d_model),
)
for p in teacher.parameters():
    p.requires_grad_(False)

def target(x):
    with torch.no_grad():
        return teacher(x)

Xtr, Xte = torch.randn(n_train, d_model), torch.randn(n_test, d_model)
Ytr, Yte = target(Xtr), target(Xte)
mu, sd = Ytr.mean(), Ytr.std()
Ytr, Yte = (Ytr - mu) / sd, (Yte - mu) / sd

def n_params(m):
    return sum(p.numel() for p in m.parameters())

def train_eval(model_fn, seed=1):
    torch.manual_seed(seed)
    net = model_fn()
    opt = torch.optim.Adam(net.parameters(), lr=3e-3)
    for _ in range(2000):
        loss = F.mse_loss(net(Xtr), Ytr)
        opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad():
        return F.mse_loss(net(Xte), Yte).item(), n_params(net)

relu_loss, relu_p = train_eval(lambda: ReLUFFN(d_model))
swiglu_loss, swiglu_p = train_eval(lambda: SwiGLUFFN(d_model))

print(f"ReLU-FFN   (hidden={4*d_model}): params={relu_p:,}  test_loss={relu_loss:.4f}")
print(f"SwiGLU-FFN (hidden={swiglu_hidden(d_model)}): params={swiglu_p:,}  test_loss={swiglu_loss:.4f}")
print(f"param ratio SwiGLU/ReLU = {swiglu_p / relu_p:.3f}  (~1.0 = matched)")
print("Honest read: no toy win for SwiGLU here; its benefit is at-scale, not toy-visible.")

## 3. Sanity checks

In [ ]:
# 1) output shape == input shape
ffn = SwiGLUFFN(64); x = torch.randn(8, 16, 64)
assert ffn(x).shape == x.shape

# 2) param count SwiGLU(8d/3) ~= ReLU(4d)
sw, re = SwiGLUFFN(64), ReLUFFN(64)
ratio = n_params(sw) / n_params(re)
assert 0.9 <= ratio <= 1.1, ratio

# 3) gate actually gates: zero gate weight -> output 0 (Swish(0)=0)
ffn = SwiGLUFFN(64)
with torch.no_grad():
    ffn.gate.weight.zero_()
out = ffn(torch.randn(4, 64))
assert torch.allclose(out, torch.zeros_like(out), atol=1e-6)

# 4) our Swish matches F.silu
x = torch.randn(1000)
assert torch.allclose(x * torch.sigmoid(x), F.silu(x), atol=1e-6)

# 5) gradient flows to gate, up, down
ffn = SwiGLUFFN(64); ffn(torch.randn(4, 64)).sum().backward()
for name in ("gate", "up", "down"):
    g = getattr(ffn, name).weight.grad
    assert g is not None and g.abs().sum() > 0, name

# 6) gate makes FFN nonlinear: f(2x) != 2 f(x)
ffn = SwiGLUFFN(64); x = torch.randn(4, 64)
assert not torch.allclose(ffn(2 * x), 2 * ffn(x), atol=1e-3)

print("All sanity checks passed.")

## 4. Reference solution (peek only after trying)

In [ ]:
class SwiGLUFFNSolution(nn.Module):
    def __init__(self, d_model, hidden=None, ffn_mult=4):
        super().__init__()
        if hidden is None:
            hidden = swiglu_hidden(d_model, ffn_mult)
        self.gate = nn.Linear(d_model, hidden, bias=False)
        self.up = nn.Linear(d_model, hidden, bias=False)
        self.down = nn.Linear(hidden, d_model, bias=False)

    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.up(x))